In [12]:
import pandas as pd
import igraph
import pickle as pk
import os

In [13]:
print("Available datasets:")
for d in set( [d.split('_')[0] for d in os.listdir("data/")] ):
    print("-", d)

Available datasets:
- Science
- LitLangMed
- Humanities


In [14]:
dataset = 'LitLangMed'

In [15]:
df = pd.read_csv( "data/" + dataset + "_enriched.csv", dtype={43:str, 44:str, 45:str, 46:str} )
print( df.shape )
print( df.columns )

(7017, 60)
Index(['School', 'Folder', 'Document Name', 'Thesis Title', 'Thesis Author',
       'Citation ID', 'Citation Type', 'Citation Title', 'Citation Authors',
       'Citation Editors', 'Citation Year', 'Journal', 'Publisher',
       'Citation Source', 'Place', 'Series', 'Volume', 'Issue', 'Pages', 'DOI',
       'Reference URL', 'Note', '_is_duplicate_within_thesis', '_is_noise',
       '_noise_reason', '_citation_type_original', 'eligible_for_matching',
       '_ineligible_reason', 'Thesis Year', 'Citation Type (refined)',
       'Citation Author Count', 'Citation Is Single Author', 'ref_id',
       'openalex_status', 'openalex_affiliations', 'openalex_countries',
       'openalex_author_breakdown', 'openalex_author_country_pairs',
       'openalex_matched_author_count', 'openalex_main_author_matched',
       'openalex_distinct_author_countries',
       'openalex_is_international_collab', 'pyorcid_status',
       'pyorcid_affiliations', 'pyorcid_countries', 'pyorcid_author_break

In [16]:
# Drop rows with nan authors
df = df.dropna(subset=['Citation Authors'])
len(df)

6428

In [17]:
def create_graph_data(df):
    
    ## CREATE NODES

    # Get unique phd documents based on doc name
    df_phds = df.drop_duplicates(subset=['Document Name'])[['Document Name', 'School']]
    df_phds.columns = ['title', 'school']
    df_phds['type'] = 'Wits PhD'

    # Get unique refs based on title, authors and ref type
    df_refs = df.drop_duplicates(subset=['Citation Authors'])[['Citation Authors', 'Citation Type']]
    df_refs.columns = ['authors', 'type']

    # Concatenate into single node df and generate unique id for each node
    df_nodes = pd.concat( [
        df_phds, 
        df_refs
    ], ignore_index = True).reset_index(names='id')

    
    ## CREATE EDGES

    # Create mappings from phd docs and refs to node ids
    phd_map = df_nodes[df_nodes['type'] == 'Wits PhD'].set_index('title')['id']    
    ref_map = df_nodes[df_nodes['type'] != 'Wits PhD'].set_index(['authors'])['id']

    # Create edges by mapping phd doc names into source column, and ref title, author and data into target column
    df_edges = pd.DataFrame({
        'source': df['Document Name'].map(phd_map),
        'target': df['Citation Authors'].map(ref_map)
    })

    return df_nodes, df_edges

def describe(g):
    print( g.vcount() )
    print( g.ecount() )
    print( g.vertex_attributes() )
    print( g.edge_attributes() )

In [18]:
nodes, edges = create_graph_data(df)

In [19]:
g = igraph.Graph.DataFrame(edges, directed=True, vertices=nodes, use_vids=False)
describe(g)

5297
6428
['name', 'title', 'school', 'type', 'authors']
[]


In [20]:
## Write graph to pickle
g.write_pickle( "graphs/" + dataset + "_graph.pk")